# Adzuna Job Fetcher

This notebook provides a more stable workflow for matching Companies House company names to Adzuna job data.

Features:
- company-name standardization
- alias-key matching
- full batch processing
- SQLite cache to avoid repeated API calls
- retry / backoff for transient failures
- checkpoint CSV outputs
- company-level summary features

**Note:** Keep your Adzuna credentials in environment variables rather than hardcoding them into the notebook.

In [1]:
import os
import re
import json
import time
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Iterable

import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [3]:
INPUT_PATH = Path("../output/UKcompanies_8_sectors_cleaned.csv") 
OUTPUT_DIR = Path("../output/adzuna_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_JOBS_CACHE_PATH = OUTPUT_DIR / "adzuna_jobs_cache.sqlite"
RAW_JOBS_CSV_PATH = OUTPUT_DIR / "adzuna_jobs_raw.csv"
SUMMARY_CSV_PATH = OUTPUT_DIR / "adzuna_company_summary.csv"

# Credentials: set these in environment variables
#   ADZUNA_APP_ID
#   ADZUNA_APP_KEY
ADZUNA_APP_ID = "9d40f51d"
ADZUNA_APP_KEY = "6a151e5f23976ded9a47ce1cf059df04"

COUNTRY = "gb"
RESULTS_PER_PAGE = 50
MAX_PAGES_PER_QUERY = 2          # keep this modest to control time/cost
MAX_QUERIES_PER_COMPANY = 3      # first 3 variants are usually enough
REQUEST_SLEEP_SECONDS = 0.4      # light pacing
MIN_QUERY_LENGTH = 2

if not ADZUNA_APP_ID or not ADZUNA_APP_KEY:
    print("Warning: ADZUNA_APP_ID / ADZUNA_APP_KEY not found in environment variables.")


In [4]:
# Load input
df = pd.read_csv(INPUT_PATH, dtype=str)

required_cols = ["CompanyNumber", "CompanyName"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df.head()


,CompanyNumber,CompanyName,CompanyStatus,CompanyCategory,CountryOfOrigin,RegAddress_Country,RegAddress_PostTown,RegAddress_PostCode,IncorporationDate,primary_sic_code,...,sector_id,Accounts_AccountCategory,Accounts_LastMadeUpDate,Accounts_NextDueDate,Mortgages_NumMortOutstanding,Mortgages_NumMortCharges,has_outstanding_charges,company_age_years,CountryOfOrigin_clean,is_uk_company
0,00000086,KENTSTONE PROPERTIES LIMITED,Active,Private Limited Company,United Kingdom,NaN,ASHFORD,TN25 6SX,1862-11-27,68209,...,4.0,SMALL,2025-03-31,2026-12-31,19,75,True,163.6,united kingdom,True
1,00000118,ASHFORD CATTLE MARKET COMPANY LIMITED(THE),Active,Private Limited Company,United Kingdom,NaN,KENT,TN23 1DA,1856-09-25,46110,...,3.0,FULL,2025-07-31,2027-04-30,0,2,False,169.7,united kingdom,True
2,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",Active,Private Limited Company,United Kingdom,NaN,LONDON,EC4A 2EA,1856-09-26,70100,...,5.0,TOTAL EXEMPTION FULL,2025-06-30,2027-03-31,2,4,True,169.7,united kingdom,True
3,00000140,N & C BUILDING PRODUCTS LIMITED,Active,Private Limited Company,United Kingdom,NaN,ROMFORD,RM8 1SP,1856-09-30,20301,...,2.0,FULL,2024-12-31,2026-09-30,1,5,True,169.7,united kingdom,True
4,00000295,METHODIST NEWSPAPER COMPANY LIMITED,Active,Private Limited Company,United Kingdom,ENGLAND,HUNSTANTON,PE36 6JG,1863-03-13,58130,...,5.0,TOTAL EXEMPTION FULL,2026-03-22,2027-12-22,1,2,True,163.3,united kingdom,True


In [21]:
# Company-name standardization

LEGAL_SUFFIXES = [
    "public limited company",
    "limited",
    "ltd",
    "plc",
    "llp",
    "lp",
    "incorporated",
    "inc",
    "corporation",
    "corp",
    "company",
    "co",
    "group",
    "holdings",
    "holding",
    "societas",
]

STOPWORDS = {"the", "and", "of", "for", "in", "on", "at", "by", "with", "from"}

def normalize_company_name(name: str) -> str:
    if pd.isna(name):
        return pd.NA

    s = str(name).strip().lower()
    s = re.sub(r"[^\w\s&]", " ", s)
    s = s.replace("&", " and ")
    s = re.sub(r"\s+", " ", s).strip()

    # strip suffixes repeatedly from the end
    changed = True
    while changed:
        changed = False
        for suf in sorted(LEGAL_SUFFIXES, key=len, reverse=True):
            new_s = re.sub(rf"(\b{re.escape(suf)}\b)$", "", s).strip()
            if new_s != s:
                s = re.sub(r"\s+", " ", new_s).strip()
                changed = True

    # remove generic stopwords
    tokens = [t for t in s.split() if t not in STOPWORDS]
    s = " ".join(tokens).strip()

    # final suffix strip after stopword cleanup
    changed = True
    while changed:
        changed = False
        for suf in sorted(LEGAL_SUFFIXES, key=len, reverse=True):
            new_s = re.sub(rf"(\b{re.escape(suf)}\b)$", "", s).strip()
            if new_s != s:
                s = re.sub(r"\s+", " ", new_s).strip()
                changed = True

    return s if s else pd.NA

def make_company_alias_key(name: str) -> str:
    cleaned = normalize_company_name(name)
    if pd.isna(cleaned):
        return pd.NA
    return re.sub(r"[^a-z0-9]+", "", str(cleaned).lower())

# def build_search_variants(name: str) -> list[str]:
#     variants = []
#     raw = str(name).strip()
#     cleaned = normalize_company_name(name)

#     if raw:
#         variants.append(raw)
#     if pd.notna(cleaned):
#         variants.append(cleaned)
#         variants.append(cleaned.title())

#     out = []
#     seen = set()
#     for x in variants:
#         x2 = re.sub(r"\s+", " ", str(x).strip())
#         if x2 and x2.lower() not in seen:
#             out.append(x2)
#             seen.add(x2.lower())
#     return out

def build_search_variants(name: str) -> list[str]:
    variants = []

    raw = str(name).strip()
    cleaned = normalize_company_name(name)

    # 最重要：先放清洗后的短名
    if pd.notna(cleaned):
        variants.append(cleaned)
        variants.append(cleaned.title())

        # 再加一个更短的核心词版本
        tokens = cleaned.split()
        if len(tokens) >= 2:
            variants.append(" ".join(tokens[:2]))   # 前两个词
        if len(tokens) >= 3:
            variants.append(" ".join(tokens[:3]))   # 前三个词

    # 原始名最后放
    if raw:
        variants.append(raw)

    # 去重
    out = []
    seen = set()
    for x in variants:
        x2 = re.sub(r"\s+", " ", str(x).strip())
        if x2 and x2.lower() not in seen:
            out.append(x2)
            seen.add(x2.lower())
    return out

df["company_name_clean"] = df["CompanyName"].apply(normalize_company_name)
df["company_alias_key"] = df["CompanyName"].apply(make_company_alias_key)
df["search_variants"] = df["CompanyName"].apply(build_search_variants)
df[["CompanyName", "company_name_clean", "company_alias_key", "search_variants"]].head(5)


,CompanyName,company_name_clean,company_alias_key,search_variants
0,KENTSTONE PROPERTIES LIMITED,kentstone properties,kentstoneproperties,"[kentstone properties, KENTSTONE PROPERTIES LI..."
1,ASHFORD CATTLE MARKET COMPANY LIMITED(THE),ashford cattle market,ashfordcattlemarket,"[ashford cattle market, ashford cattle, ASHFOR..."
2,"ORIENTAL GAS COMPANY, LIMITED(THE)",oriental gas,orientalgas,"[oriental gas, ORIENTAL GAS COMPANY, LIMITED(T..."
3,N & C BUILDING PRODUCTS LIMITED,n c building products,ncbuildingproducts,"[n c building products, n c, n c building, N &..."
4,METHODIST NEWSPAPER COMPANY LIMITED,methodist newspaper,methodistnewspaper,"[methodist newspaper, METHODIST NEWSPAPER COMP..."


In [22]:
# Export
keep_cols = [
    c for c in [
        "CompanyNumber",
        "CompanyName",
        "company_name_raw",
        "company_name_clean",
        "company_alias_key",
        "search_variants",
        "primary_sector",
        "sector_id",
        "CountryOfOrigin",
        "RegAddress_Country",
        "is_uk_company",
    ]
    if c in df.columns
]

df_out = df[keep_cols].copy()
df_out.to_csv("../output/standard_companies.csv", index=False, encoding="utf-8-sig")

In [23]:
BASE_URL = "https://api.adzuna.com/v1/api/jobs"
RESULTS_PER_PAGE = 50
MAX_PAGES_PER_QUERY = 2
MAX_QUERIES_PER_COMPANY = 3
SLEEP_SECONDS = 0.4


RAW_JOBS_CSV = OUTPUT_DIR / "adzuna_jobs_raw.csv"
FILTERED_JOBS_CSV = OUTPUT_DIR / "adzuna_jobs_filtered.csv"
SUMMARY_CSV = OUTPUT_DIR / "adzuna_company_summary.csv"
CACHE_DB = OUTPUT_DIR / "adzuna_cache.sqlite"

def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

session = make_session()

def init_cache():
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS adzuna_cache (
            cache_key TEXT PRIMARY KEY,
            response_json TEXT,
            fetched_at TEXT
        )
    """)
    conn.commit()
    conn.close()

def cache_key(company_number: str, query: str, where: Optional[str], page: int) -> str:
    return f"{company_number}|{query.lower().strip()}|{(where or '').lower().strip()}|{page}"

def cache_get(key: str) -> Optional[dict]:
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("SELECT response_json FROM adzuna_cache WHERE cache_key = ?", (key,))
    row = cur.fetchone()
    conn.close()
    if not row:
        return None
    try:
        return json.loads(row[0])
    except Exception:
        return None

def cache_put(key: str, response: dict):
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        INSERT OR REPLACE INTO adzuna_cache (cache_key, response_json, fetched_at)
        VALUES (?, ?, datetime('now'))
    """, (key, json.dumps(response, ensure_ascii=False)))
    conn.commit()
    conn.close()

def adzuna_search_jobs(query: str, where: Optional[str] = None, page: int = 1, results_per_page: int = 50) -> dict:
    url = f"{BASE_URL}/{COUNTRY}/search/{page}"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query,
        "results_per_page": results_per_page,
        "content-type": "application/json",
    }
    if where:
        params["where"] = where

    resp = session.get(url, params=params, timeout=40)
    resp.raise_for_status()
    return resp.json()

init_cache()

In [ ]:
# Adzuna session with retries
# def make_session() -> requests.Session:
#     session = requests.Session()
#     retry = Retry(
#         total=5,
#         read=5,
#         connect=5,
#         backoff_factor=1.5,
#         status_forcelist=[429, 500, 502, 503, 504],
#         allowed_methods=frozenset(["GET"]),
#         raise_on_status=False,
#     )
#     adapter = HTTPAdapter(max_retries=retry)
#     session.mount("https://", adapter)
#     session.mount("http://", adapter)
#     return session

# session = make_session()

# def adzuna_search_jobs(query: str, where: Optional[str] = None, page: int = 1, results_per_page: int = 50) -> dict:
#     """
#     Search Adzuna jobs by query.
#     """
#     if not ADZUNA_APP_ID or not ADZUNA_APP_KEY:
#         raise ValueError("Set ADZUNA_APP_ID and ADZUNA_APP_KEY in environment variables.")

#     url = f"https://api.adzuna.com/v1/api/jobs/{COUNTRY}/search/{page}"
#     params = {
#         "app_id": ADZUNA_APP_ID,
#         "app_key": ADZUNA_APP_KEY,
#         "results_per_page": results_per_page,
#         "what": query,
#         "content-type": "application/json",
#     }
#     if where:
#         params["where"] = where

#     resp = session.get(url, params=params, timeout=40)
#     resp.raise_for_status()
#     return resp.json()


In [44]:
def alias_score(company_key: Optional[str], display_key: Optional[str]) -> bool:
    """
    简单容错匹配：
    - 一方为空就返回 False
    - 一方包含另一方就认为可能匹配
    """
    if pd.isna(company_key) or pd.isna(display_key):
        return False

    company_key = str(company_key).strip()
    display_key = str(display_key).strip()

    if not company_key or not display_key:
        return False

    return company_key in display_key or display_key in company_key

def fetch_jobs_for_company(
    company_number: str,
    company_name: str,
    where: Optional[str] = "uk",
    max_pages: int = MAX_PAGES_PER_QUERY,
    max_queries: int = MAX_QUERIES_PER_COMPANY,
    use_cache: bool = True,
    keep_unfiltered: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    返回：
    1) raw_df: Adzuna 原始命中（不做公司名过滤）
    2) filtered_df: 经过公司名 alias 匹配后的结果
    """
    company_key = make_company_alias_key(company_name)
    variants = build_search_variants(company_name)[:max_queries]

    raw_rows = []
    filtered_rows = []
    seen_job_ids = set()

    for q in variants:
        if len(str(q).strip()) < 2:
            continue

        print(f"Querying: {company_name} | variant={q}")

        for page in range(1, max_pages + 1):
            key = cache_key(company_number, q, where, page)

            data = cache_get(key) if use_cache else None
            if data is None:
                try:
                    data = adzuna_search_jobs(q, where=where, page=page, results_per_page=RESULTS_PER_PAGE)
                    if use_cache:
                        cache_put(key, data)
                except Exception as e:
                    raw_rows.append({
                        "company_number": company_number,
                        "company_name_input": company_name,
                        "query": q,
                        "page": page,
                        "error": str(e),
                        "match_type": "api_error",
                    })
                    break

                time.sleep(0.4)

            results = data.get("results", []) if isinstance(data, dict) else []
            total = data.get("count", None) if isinstance(data, dict) else None

            # 先把原始返回都记下来
            for item in results:
                job_id = item.get("id")
                if job_id and job_id in seen_job_ids:
                    continue

                company_display = (item.get("company") or {}).get("display_name", "")
                company_display_key = make_company_alias_key(company_display)

                base_row = {
                    "company_number": company_number,
                    "company_name_input": company_name,
                    "company_name_clean": normalize_company_name(company_name),
                    "company_alias_key": company_key,
                    "query": q,
                    "page": page,
                    "adzuna_total": total,
                    "job_id": job_id,
                    "job_title": item.get("title"),
                    "company_display_name": company_display,
                    "company_display_alias_key": company_display_key,
                    "location": (item.get("location") or {}).get("display_name"),
                    "location_area": " > ".join((item.get("location") or {}).get("area", [])) if (item.get("location") or {}).get("area") else None,
                    "created": item.get("created"),
                    "modified": item.get("modified"),
                    "redirect_url": item.get("redirect_url"),
                    "salary_min": item.get("salary_min"),
                    "salary_max": item.get("salary_max"),
                    "contract_type": item.get("contract_type"),
                    "contract_time": item.get("contract_time"),
                    "category": (item.get("category") or {}).get("label"),
                }
                raw_rows.append({**base_row, "match_type": "raw"})

                # 再做过滤结果
                if alias_score(company_key, company_display_key):
                    filtered_rows.append({**base_row, "match_type": "filtered"})
                    seen_job_ids.add(job_id)

            if len(results) < RESULTS_PER_PAGE:
                break

    raw_df = pd.DataFrame(raw_rows)
    filtered_df = pd.DataFrame(filtered_rows)

    return raw_df, filtered_df

In [45]:
def fetch_jobs_for_company_debug(company_number: str, company_name: str, where: str | None = "uk"):
    company_key = make_company_alias_key(company_name)
    variants = build_search_variants(company_name)[:4]

    all_rows = []

    for q in variants:
        print(f"Querying: {company_name} | variant={q}")

        for page in range(1, 3):
            data = adzuna_search_jobs(q, where=where, page=page, results_per_page=50)

            total = data.get("count")
            results = data.get("results", [])
            print(f"  page={page}, total={total}, results={len(results)}")

            for item in results:
                company_display = (item.get("company") or {}).get("display_name", "")
                company_display_key = make_company_alias_key(company_display)

                # 看原始返回
                all_rows.append({
                    "company_number": company_number,
                    "company_name_input": company_name,
                    "query": q,
                    "page": page,
                    "job_id": item.get("id"),
                    "job_title": item.get("title"),
                    "company_display_name": company_display,
                    "company_display_alias_key": company_display_key,
                    "location": (item.get("location") or {}).get("display_name"),
                    "created": item.get("created"),
                    "redirect_url": item.get("redirect_url"),
                })

            if len(results) < 50:
                break

        if all_rows:
            break

    return pd.DataFrame(all_rows)

In [46]:
test = fetch_jobs_for_company_debug("00000121", "ORIENTAL GAS COMPANY")
test.head()

Querying: ORIENTAL GAS COMPANY | variant=oriental gas
  page=1, total=2, results=2


,company_number,company_name_input,query,page,job_id,job_title,company_display_name,company_display_alias_key,location,created,redirect_url
0,00000121,ORIENTAL GAS COMPANY,oriental gas,1,5210480985,Gas Turbine R&D - Mechanical Design Engineer,Siemens,siemens,UK,2025-05-22T08:54:31Z,https://www.adzuna.co.uk/jobs/details/52104809...
1,00000121,ORIENTAL GAS COMPANY,oriental gas,1,5628045697,Senior Gameplay and Animation Engineer,2K Czech,2kczech,"Brighton, East Sussex",2026-02-13T17:23:36Z,https://www.adzuna.co.uk/jobs/details/56280456...


In [47]:
# 按行业先挑更像雇佣主体的公司
employer_like_sectors = {
    "Technology, legal & professional",
    "Healthcare",
    "Manufacturing",
    "Wholesale & Retail",
    "Fast growth & emerging sector",
    "Public sector, education & charities",
}

# by primary_sector
company_seed = (
    df[df["primary_sector"].isin(employer_like_sectors)]
    .copy()
)

# 测试
company_seed_10 = company_seed[["CompanyNumber", "CompanyName"]].drop_duplicates().head(10).copy()
company_seed_10

,CompanyNumber,CompanyName
1,00000118,ASHFORD CATTLE MARKET COMPANY LIMITED(THE)
2,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)"
3,00000140,N & C BUILDING PRODUCTS LIMITED
4,00000295,METHODIST NEWSPAPER COMPANY LIMITED
5,00000371,LONDON AND SUBURBAN LAND AND BUILDING COMPANY ...
8,00000687,BRITISH INDIAN TEA COMPANY LIMITED
9,00000866,STAVELEY INDUSTRIES LIMITED
12,00001812,QUANTUM CLOTHING GROUP LIMITED
13,00001978,GROSVENOR ESTATES LIMITED
14,00002120,WHITBREAD INTERNATIONAL LIMITED


In [48]:
def run_batch(companies_df: pd.DataFrame, where: str = "uk"):
    all_raw = []
    all_filtered = []

    for i, row in enumerate(companies_df.itertuples(index=False), start=1):
        company_number = getattr(row, "CompanyNumber")
        company_name = getattr(row, "CompanyName")

        print(f"\n[{i}/{len(companies_df)}] Fetching: {company_name}")

        raw_df, filtered_df = fetch_jobs_for_company(
            company_number=str(company_number),
            company_name=str(company_name),
            where=where,
            max_pages=MAX_PAGES_PER_QUERY,
            max_queries=MAX_QUERIES_PER_COMPANY,
            use_cache=True,
        )

        if not raw_df.empty:
            all_raw.append(raw_df)
        if not filtered_df.empty:
            all_filtered.append(filtered_df)

        # 每家公司跑完写缓存，防止中断丢数据
        if not raw_df.empty:
            header = not RAW_JOBS_CSV.exists()
            raw_df.to_csv(RAW_JOBS_CSV, mode="a", header=header, index=False, encoding="utf-8-sig")
        if not filtered_df.empty:
            header = not FILTERED_JOBS_CSV.exists()
            filtered_df.to_csv(FILTERED_JOBS_CSV, mode="a", header=header, index=False, encoding="utf-8-sig")

    raw_all = pd.concat(all_raw, ignore_index=True) if all_raw else pd.DataFrame()
    filtered_all = pd.concat(all_filtered, ignore_index=True) if all_filtered else pd.DataFrame()

    return raw_all, filtered_all

In [49]:
raw_10, filtered_10 = run_batch(company_seed_10, where="uk")

print("raw shape:", raw_10.shape)
print("filtered shape:", filtered_10.shape)

raw_10.head()


[1/10] Fetching: ASHFORD CATTLE MARKET COMPANY LIMITED(THE)
Querying: ASHFORD CATTLE MARKET COMPANY LIMITED(THE) | variant=ashford cattle market
Querying: ASHFORD CATTLE MARKET COMPANY LIMITED(THE) | variant=ashford cattle
Querying: ASHFORD CATTLE MARKET COMPANY LIMITED(THE) | variant=ASHFORD CATTLE MARKET COMPANY LIMITED(THE)

[2/10] Fetching: ORIENTAL GAS COMPANY, LIMITED(THE)
Querying: ORIENTAL GAS COMPANY, LIMITED(THE) | variant=oriental gas
Querying: ORIENTAL GAS COMPANY, LIMITED(THE) | variant=ORIENTAL GAS COMPANY, LIMITED(THE)

[3/10] Fetching: N & C BUILDING PRODUCTS LIMITED
Querying: N & C BUILDING PRODUCTS LIMITED | variant=n c building products
Querying: N & C BUILDING PRODUCTS LIMITED | variant=n c
Querying: N & C BUILDING PRODUCTS LIMITED | variant=n c building

[4/10] Fetching: METHODIST NEWSPAPER COMPANY LIMITED
Querying: METHODIST NEWSPAPER COMPANY LIMITED | variant=methodist newspaper
Querying: METHODIST NEWSPAPER COMPANY LIMITED | variant=METHODIST NEWSPAPER COMPANY 

,company_number,company_name_input,company_name_clean,company_alias_key,query,page,adzuna_total,job_id,job_title,company_display_name,...,location_area,created,modified,redirect_url,salary_min,salary_max,contract_type,contract_time,category,match_type
0,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",oriental gas,orientalgas,oriental gas,1,2,5210480985,Gas Turbine R&D - Mechanical Design Engineer,Siemens,...,UK,2025-05-22T08:54:31Z,None,https://www.adzuna.co.uk/jobs/details/52104809...,43354.18,43354.18,None,full_time,Engineering Jobs,raw
1,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",oriental gas,orientalgas,oriental gas,1,2,5628045697,Senior Gameplay and Animation Engineer,2K Czech,...,UK > South East England > East Sussex > Brighton,2026-02-13T17:23:36Z,None,https://www.adzuna.co.uk/jobs/details/56280456...,62670.89,62670.89,contract,None,Engineering Jobs,raw
2,00000140,N & C BUILDING PRODUCTS LIMITED,n c building products,ncbuildingproducts,n c,1,99,5744694334,Engineering Professionals N.e.c,Pertemps North Midlands,...,UK > Yorkshire And The Humber > North Yorkshir...,2026-05-29T18:03:48Z,None,https://www.adzuna.co.uk/jobs/land/ad/57446943...,36000.00,40000.00,permanent,full_time,Engineering Jobs,raw
3,00000140,N & C BUILDING PRODUCTS LIMITED,n c building products,ncbuildingproducts,n c,1,99,5766992969,"Marketing, Sales And Advertising Directors N.e.c",West Midlands and Worcestershire Perm Hub,...,UK > West Midlands > Birmingham > Balsall Heath,2026-06-17T21:07:40Z,None,https://www.adzuna.co.uk/jobs/land/ad/57669929...,30000.00,32000.00,None,full_time,"PR, Advertising & Marketing Jobs",raw
4,00000140,N & C BUILDING PRODUCTS LIMITED,n c building products,ncbuildingproducts,n c,1,99,5768207383,"Marketing, Sales And Advertising Directors N.e.c.",West Midlands & Worcestershire Perm Hub,...,UK > West Midlands > Birmingham,2026-06-18T16:45:52Z,None,https://www.adzuna.co.uk/jobs/details/57682073...,30000.00,32000.00,permanent,full_time,"PR, Advertising & Marketing Jobs",raw


In [50]:
def build_company_summary(jobs_df: pd.DataFrame) -> pd.DataFrame:
    if jobs_df.empty:
        return pd.DataFrame()

    out = jobs_df.copy()
    out["created"] = pd.to_datetime(out["created"], errors="coerce")

    summary = (
        out.groupby(["company_number", "company_name_input"], dropna=False)
        .agg(
            vacancy_count=("job_id", lambda s: s.nunique(dropna=True)),
            latest_job_date=("created", "max"),
            earliest_job_date=("created", "min"),
            unique_locations=("location", lambda s: s.nunique(dropna=True)),
            unique_job_titles=("job_title", lambda s: s.nunique(dropna=True)),
        )
        .reset_index()
    )

    summary["job_span_days"] = (
        summary["latest_job_date"] - summary["earliest_job_date"]
    ).dt.days

    return summary

summary_10 = build_company_summary(raw_10)
summary_10.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

summary_10.head()

,company_number,company_name_input,vacancy_count,latest_job_date,earliest_job_date,unique_locations,unique_job_titles,job_span_days
0,00000121,"ORIENTAL GAS COMPANY, LIMITED(THE)",2,2026-02-13 17:23:36+00:00,2025-05-22 08:54:31+00:00,2,2,267
1,00000140,N & C BUILDING PRODUCTS LIMITED,99,2026-06-21 16:40:31+00:00,2024-07-31 15:13:13+00:00,56,79,690
2,00000371,LONDON AND SUBURBAN LAND AND BUILDING COMPANY ...,4,2026-06-21 15:30:40+00:00,2025-10-08 11:21:31+00:00,4,3,256
3,00000687,BRITISH INDIAN TEA COMPANY LIMITED,5,2026-06-15 10:51:59+00:00,2026-03-14 01:28:50+00:00,2,4,93


In [51]:
raw_10[
    [
        "company_name_input",
        "company_display_name",
        "job_title"
    ]
].head(30)

,company_name_input,company_display_name,job_title
0,"ORIENTAL GAS COMPANY, LIMITED(THE)",Siemens,Gas Turbine R&D - Mechanical Design Engineer
1,"ORIENTAL GAS COMPANY, LIMITED(THE)",2K Czech,Senior Gameplay and Animation Engineer
2,N & C BUILDING PRODUCTS LIMITED,Pertemps North Midlands,Engineering Professionals N.e.c
3,N & C BUILDING PRODUCTS LIMITED,West Midlands and Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c"
4,N & C BUILDING PRODUCTS LIMITED,West Midlands & Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c."
5,N & C BUILDING PRODUCTS LIMITED,Nicholls & Clarke Limited,Class 2 HGV driver
6,N & C BUILDING PRODUCTS LIMITED,Thales UK Limited,Machine Technician
7,N & C BUILDING PRODUCTS LIMITED,Recruit4staff,CNC Programmer
8,N & C BUILDING PRODUCTS LIMITED,GE Aerospace,Dowty Field Service Engineer
9,N & C BUILDING PRODUCTS LIMITED,Kroll,Loan Portfolio Transaction Manager


In [52]:
raw_10[
    ["company_name_input","company_display_name"]
].drop_duplicates()

,company_name_input,company_display_name
0,"ORIENTAL GAS COMPANY, LIMITED(THE)",Siemens
1,"ORIENTAL GAS COMPANY, LIMITED(THE)",2K Czech
2,N & C BUILDING PRODUCTS LIMITED,Pertemps North Midlands
3,N & C BUILDING PRODUCTS LIMITED,West Midlands and Worcestershire Perm Hub
4,N & C BUILDING PRODUCTS LIMITED,West Midlands & Worcestershire Perm Hub
...,...,...
100,N & C BUILDING PRODUCTS LIMITED,Carmarthenshire County Council
103,LONDON AND SUBURBAN LAND AND BUILDING COMPANY ...,Reeson Education
104,LONDON AND SUBURBAN LAND AND BUILDING COMPANY ...,Marchant Recruitment
107,BRITISH INDIAN TEA COMPANY LIMITED,Kricket Canary Wharf


In [57]:
raw_10[
    raw_10["company_name_input"] == "N & C BUILDING PRODUCTS LIMITED"
][
    [
        "query",
        "company_display_name",
        "job_title"
    ]
].head(50)

,query,company_display_name,job_title
2,n c,Pertemps North Midlands,Engineering Professionals N.e.c
3,n c,West Midlands and Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c"
4,n c,West Midlands & Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c."
5,n c,Nicholls & Clarke Limited,Class 2 HGV driver
6,n c,Thales UK Limited,Machine Technician
7,n c,Recruit4staff,CNC Programmer
8,n c,GE Aerospace,Dowty Field Service Engineer
9,n c,Kroll,Loan Portfolio Transaction Manager
10,n c,Kroll,"Portfolio Transaction Manager, Agency and Trus..."
11,n c,The Best Connection,CNC Operator


In [58]:
raw_10[
    [
        "query",
        "company_display_name",
        "job_title"
    ]
].head(50)

,query,company_display_name,job_title
0,oriental gas,Siemens,Gas Turbine R&D - Mechanical Design Engineer
1,oriental gas,2K Czech,Senior Gameplay and Animation Engineer
2,n c,Pertemps North Midlands,Engineering Professionals N.e.c
3,n c,West Midlands and Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c"
4,n c,West Midlands & Worcestershire Perm Hub,"Marketing, Sales And Advertising Directors N.e.c."
5,n c,Nicholls & Clarke Limited,Class 2 HGV driver
6,n c,Thales UK Limited,Machine Technician
7,n c,Recruit4staff,CNC Programmer
8,n c,GE Aerospace,Dowty Field Service Engineer
9,n c,Kroll,Loan Portfolio Transaction Manager
